In [8]:
import os
import requests
from dotenv import load_dotenv

In [9]:
load_dotenv(override=True)

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

print("user set:", bool(pushover_user))
print("token set:", bool(pushover_token))

user set: True
token set: True


In [10]:
def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    response = requests.post(pushover_url, data=payload)
    print("Status:", response.status_code)

In [12]:
push("Hello i am siva")

Push: Hello i am siva
Status: 200


In [13]:
push("Hello i am SAI")

Push: Hello i am SAI
Status: 200


## two tool functions

In [14]:
def record_user_details(email, name="Name not provided", notes="not provided"):
    push(f"Recording interest from {name} with email {email} and notes {notes}")
    return "OK"

In [15]:
def record_unknown_question(question):
    push(f"Recording question I couldn't answer: {question}")
    return "OK"

In [16]:
record_user_details("test@example.com", name="Test Person", notes="Testing the tool")
record_unknown_question("What is your favourite film?")

Push: Recording interest from Test Person with email test@example.com and notes Testing the tool
Status: 200
Push: Recording question I couldn't answer: What is your favourite film?
Status: 200


'OK'

In [19]:
#both tools to the model as JSON
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a visitor wants to be in touch and has provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this visitor",
            },
            "name": {
                "type": "string",
                "description": "The visitor's name, if they provided it",
            },
            "notes": {
                "type": "string",
                "description": "Any extra context from the conversation worth recording",
            },
        },
        "required": ["email"],
        "additionalProperties": False,
    },
}

In [20]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Always use this tool to record any question you could not answer because you did not know the answer",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that could not be answered",
            },
        },
        "required": ["question"],
        "additionalProperties": False,
    },
}

In [21]:
tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json},
]

In [22]:
tools

[{'type': 'function',
  'function': {'name': 'record_user_details',
   'description': 'Use this tool to record that a visitor wants to be in touch and has provided an email address',
   'parameters': {'type': 'object',
    'properties': {'email': {'type': 'string',
      'description': 'The email address of this visitor'},
     'name': {'type': 'string',
      'description': "The visitor's name, if they provided it"},
     'notes': {'type': 'string',
      'description': 'Any extra context from the conversation worth recording'}},
    'required': ['email'],
    'additionalProperties': False}}},
 {'type': 'function',
  'function': {'name': 'record_unknown_question',
   'description': 'Always use this tool to record any question you could not answer because you did not know the answer',
   'parameters': {'type': 'object',
    'properties': {'question': {'type': 'string',
      'description': 'The question that could not be answered'}},
    'required': ['question'],
    'additionalPropert

In [23]:
import json

In [24]:
def handle_tool_calls_with_if(tool_calls):
    results = []

    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)
        else:
            result = "No tool found"

        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})

    return results

In [25]:
tool_functions = {
    "record_user_details": record_user_details,
    "record_unknown_question": record_unknown_question,
}

In [26]:
def handle_tool_calls(tool_calls):
    results = []

    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        tool_function = tool_functions.get(tool_name)

        if tool_function:
            result = tool_function(**arguments)
        else:
            result = "No tool found"

        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id": tool_call.id})

    return results

In [27]:
from types import SimpleNamespace

fake_call = SimpleNamespace(
    id="test-1",
    function=SimpleNamespace(
        name="record_unknown_question",
        arguments='{"question": "What is your favourite film?"}',
    ),
)

handle_tool_calls([fake_call])

Tool called: record_unknown_question
Push: Recording question I couldn't answer: What is your favourite film?
Status: 200


[{'role': 'tool', 'content': '"OK"', 'tool_call_id': 'test-1'}]